# HNSCC-3DCT-RT

Bejarano, Tatiana, Mariluz De Ornelas‐Couto, and Ivaylo B. Mihaylov. "Longitudinal fan‐beam computed tomography dataset for head‐and‐neck squamous cell carcinoma patients."  Medical physics 46.5 (2019): 2526-2537.

"Data were exported from Eclipse in DICOM (images), DICOM-RTSTRUCT (structures), and DICOM-RTDOSE (doses) format and then imported into MIM. It should be noted that we have two DICOM-RTst per imaging set, one with series description MIM structures and another named ARIA Rad Onc Structure Sets. The RTst-MIM structures was renamed by us to remain consistent with Table V structures. On the other hand, RTst-ARIA Rad Onc Structure Sets were the original structures as part of treatment planning and delivery. Our recommendation is to use RTst-MIM structures so that the information on Table V is helpful."

In [ ]:
"""
CT and RTDOSE files can be matched based on the DICOM tag (0020,0052) Frame of Reference UID
CT and RTSTRUCT files can be matched based on the DICOM tag (0020,000d) Study Instance UID + (0008,1090) Manufacturer's Model Name
"""

In [ ]:
import pickle
with open(r"C:\Users\bilel.guetarni\Desktop\SEQ-RT\data\tcia.pkl", "rb") as f:
    patients = pickle.load(f)

In [ ]:
import pydicom

txt = ""
for p in patients:
    p.sort_imaging()
    ct = p.ct[0]
    txt += str(p.id)
    if not(ct.rtstruct is None):
        dcm = pydicom.dcmread(ct.rtstruct.get_dcm_path())
        for roi in dcm.StructureSetROISequence:
            txt += f"\n\t{roi.ROIName}"
    txt += "\n\n"

with open(r"C:\Users\bilel.guetarni\Desktop\tmp\oars.txt", "w") as file:
    file.write(txt)

# plot data statistics

In [ ]:
import numpy as np
import plotly.express as px
import plotly.graph_objs as go

def plot_clinical(hpv=None, lrr=None, os=None):
    if not hpv is None:
        fig = go.Figure(data=[go.Pie(labels=hpv.unique(), values=[list(hpv.values).count(k) for k in hpv.unique()], title="HPV")])
        fig.show()

    if not lrr is None:
        fig = go.Figure(data=[go.Pie(labels=lrr.unique(), values=[list(lrr.values).count(k) for k in lrr.unique()], title="locoregional recurrence")])
        fig.show()

    if not os is None:
        x = np.sort(os.values.astype(np.int16))
        y = [x[x >= i].size/x.size for i in x]
        fig = px.line(x=x, y=y, title="overall survival")
        fig.show()

## name
HPV
Local + Regional
Distant Metastasis
RFS
OS

## Head-Neck-CT-Atlas
/
Loco-regional Control Censor ; Site of recurrence (Distal/Local/ Locoregional)
Site of recurrence (Distal/Local/ Locoregional)
Disease-free interval (months)
Survival  (months)

## Head-Neck-PET-CT
HPV status
Locoregional
Distant
Time – diagnosis to LR (days) ; Time – diagnosis to DM (days)
Time – diagnosis to Death (days)

## HNSCC-3DCT-RT
/
/
/
/
/

## Oropharyngeal-Radiomics-Outcomes
HPV Status
Locoregional control
Freedom from distant metastasis
Relapse-free survival
Overall survival_duration of Merged updated ASRM V2

## QIN-HEADNECK
/
Location of First Recurrence
Location of First Recurrence
Date of Recurrence
Date of Death

## RADCURE
HPV
Local ; Regional
Distant
Date Local ; Date Regional ; Date Distant
Date of Death

In [ ]:
import plotly.graph_objs as go
import pandas

"""
<<Duration of locoregional control was defined as the time from diagnosis to the date of locoregional recurrence. 
Distant recurrences and second primary tumors were censored.>>

https://www.nature.com/articles/sdata2018173
"""

# Head-Neck-CT-Atlas
df = pandas.read_excel(r"F:\TCIA\Head-Neck-CT-Atlas\HNSCC-MDA-Data_update_20240514.xlsx", sheet_name="HNSCC-MDA-Data_update")
print(df.columns)

# lrr = df["Loco-regional Control Censor"]
# lrr = lrr[lrr.notna()]
# lrr[lrr == 1] = "negative"
# lrr[lrr == 0] = "positive"

# c = "Survival  (months)"
# os = df[df[c].notna()][c]
# os *= 30

# plot_clinical(lrr=lrr, os=os)

rfs = df["Disease-free interval (months)"]

rfs2y = []
for _, row in df.iterrows():
    try:
        r = row["Disease-free interval (months)"]
        if r > 2*12:
            rfs2y.append(1)
        else:
            rfs2y.append(0)
    except (KeyError, TypeError):
        continue

fig = go.Figure(data=[go.Pie(labels=list(set(rfs2y)), values=[rfs2y.count(k) for k in set(rfs2y)], title="RFS 2 years")])
fig.show()

In [ ]:
import pandas

# Head-Neck-PET-CT
df = []
for i in ("HGJ", "CHUS", "HMR", "CHUM"):
    df__ = pandas.read_excel(r"F:\TCIA\Head-Neck-PET-CT\INFOclinical_HN_Version2_30may2018.xlsx", sheet_name=i)
    df__["center"] = i
    df.append(df__)
df = pandas.concat(df)
print(df.columns)

hpv = df["HPV status"]
hpv[hpv == 'N/A'] = float("nan")
hpv[hpv == '-'] = "negative"
hpv[hpv == '+'] = "positive"

# "Time – diagnosis to LR (days)"
lrr = df['Locoregional']

c = "Time – diagnosis to Death (days)"
os = df[df[c].notna()][c]

plot_clinical(hpv=hpv, lrr=lrr, os=os)

In [ ]:
import pandas

# Oropharyngeal-Radiomics-Outcomes
df = pandas.read_csv(r"F:\TCIA\Oropharyngeal-Radiomics-Outcomes\Radiomics_Outcome_Prediction_in_OPC_ASRM_corrected.csv")
print(df.columns)

hpv = df["HPV Status"]
hpv[hpv == 'Unknown'] = float("nan")
hpv[hpv == 'N'] = "negative"
hpv[hpv == 'P'] = "positive"

# "Locoregional control_duration of Merged updated ASRM V2"
lrr = df['Locoregional control']

c = "Overall survival_duration of Merged updated ASRM V2"
os = df[df[c].notna()][c]

plot_clinical(hpv=hpv, lrr=lrr, os=os)

In [ ]:
import pandas
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_recc_data(data):
    rfs2y = []
    for i in data:
        dt, cens = i["dt"], i["censored"]
        if not(cens):
            if dt < 2*365:
                rfs2y.append({"y": "yes", "year": 2})
                rfs2y.append({"y": "yes", "year": 5})
            elif 2*365 < dt and dt < 5*365:
                rfs2y.append({"y": "no", "year": 2})
                rfs2y.append({"y": "yes", "year": 5})
            else:
                rfs2y.append({"y": "no", "year": 2})
                rfs2y.append({"y": "no", "year": 5})
        else:
            if dt > 2*365:
                rfs2y.append({"y": "no", "year": 2})
            if dt > 5*365:
                rfs2y.append({"y": "no", "year": 5})
    rfs2y = pandas.DataFrame(rfs2y)

    fig = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'}, {'type':'domain'}]])
    for c, year in enumerate(rfs2y["year"].unique()):
        df = rfs2y[rfs2y["year"] == year]
        print(len(df))
        labels = df["y"].unique()
        values = [len(df[df["y"] == i]) for i in labels]
        fig.add_trace(go.Pie(labels=labels, values=values, title=f"{year} years"), 1, c+1)
    
    fig.update_layout(autosize=False, width=800, height=400)
    fig.show()


# QIN-HEADNECK
df1 = pandas.read_excel(r"F:\TCIA\QIN-HEADNECK\Batch_01-and-Batch_02-Clinical-Data_aug242020.xlsx", header=0, sheet_name="uiowa_clinical_data_batch1_aug2")
df2 = pandas.read_excel(r"F:\TCIA\QIN-HEADNECK\Batch_01-and-Batch_02-Clinical-Data_aug242020.xlsx", header=0, sheet_name="uiowa_clinical_data_batch2_aug2")
df = pandas.concat((df1, df2)).drop(index=0)
df = df[["Date of cancer recurrence", "Radiotherapy Procedure, Date treatment stopped", "Follow-up visit date"]]

data = []
for _, row in df.iterrows():
    rtend = row["Radiotherapy Procedure, Date treatment stopped"]
    if pandas.isna(rtend):
        continue

    try:
        reccdt = row["Date of cancer recurrence"]
        followup = row["Follow-up visit date"]
        if pandas.isna(reccdt):
            data.append({"dt": (followup - rtend).days, "censored": True})
        else:
            data.append({"dt": (reccdt - rtend).days, "censored": False})
    except (KeyError, TypeError):
        continue

plot_recc_data(data)

In [ ]:
import pandas

def convert_to_datetime(df, c):
    return pandas.to_datetime(df[c], format="%d/%m/%Y")

# RADCURE
df = pandas.read_excel(r"F:\TCIA\RADCURE\RADCURE_Clinical_v04_20241219 (1).xlsx")
print(df.columns)

hpv = df["HPV"]
hpv[hpv == 'Yes, Negative'] = "negative"
hpv[hpv == 'Yes, positive'] = "positive"


# lrr_valid = ['Persistent', 'Yes', 'Possible']
# lrr = df[["Local", "Regional", "Distant"]]
# lrr = lrr[lrr.isin(lrr_valid)]
# print(lrr)

c1, c2 = "RT Start", "Date of Death"
os = df[[c1, c2]]
os = os.dropna()
os[c1] = convert_to_datetime(os, c1)
os[c2] = convert_to_datetime(os, c2)
os["os"] = [(row[c2] - row[c1]).days for _, row in os.iterrows()]
os = os["os"]

# plot_clinical(hpv=hpv, lrr=lrr, os=os)
plot_clinical(hpv=hpv, os=os)

In [ ]:
import pandas
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import datetime

def plot_recc_data(data):
    rfs2y = []
    for i in data:
        dt, cens = i["dt"], i["censored"]
        if not(cens):
            if dt < 2*365:
                rfs2y.append({"y": "yes", "year": 2})
                rfs2y.append({"y": "yes", "year": 5})
            elif 2*365 < dt and dt < 5*365:
                rfs2y.append({"y": "no", "year": 2})
                rfs2y.append({"y": "yes", "year": 5})
            else:
                rfs2y.append({"y": "no", "year": 2})
                rfs2y.append({"y": "no", "year": 5})
        else:
            if dt > 2*365:
                rfs2y.append({"y": "no", "year": 2})
            if dt > 5*365:
                rfs2y.append({"y": "no", "year": 5})
    rfs2y = pandas.DataFrame(rfs2y)

    fig = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'}, {'type':'domain'}]])
    for c, year in enumerate(rfs2y["year"].unique()):
        df = rfs2y[rfs2y["year"] == year]
        print(len(df))
        labels = df["y"].unique()
        values = [len(df[df["y"] == i]) for i in labels]
        fig.add_trace(go.Pie(labels=labels, values=values, title=f"{year} years"), 1, c+1)
    
    fig.update_layout(autosize=False, width=800, height=400)
    fig.show()


# RADCURE
df = pandas.read_excel(r"F:\TCIA\RADCURE\RADCURE_Clinical_v04_20241219 (1).xlsx")

data = []
for _, row in df.iterrows():
    try:
        # add fractions to rt start date to obtain rt end date
        rtend = row["RT Start"].date() + datetime.timedelta(days=int(row["Fx"] / 5)*7)
    except ValueError:
        continue

    reccdt = filter(lambda i: not(pandas.isna(i)), row[["Date Local", "Date Regional", "Date Distant"]])
    reccdt = list(reccdt)
    if len(reccdt) == 0:
        reccdt = None
    elif len(reccdt) == 1:
        reccdt = reccdt[0].date()
    else:
        reccdt = min(*reccdt).date()

    followup = row["Last FU"].date()
    if pandas.isna(reccdt):
        dt = (followup - rtend).days
        if dt > 0:
            data.append({"dt": dt, "censored": True})
    else:
        dt = (reccdt - rtend).days
        if dt > 0:
            data.append({"dt": dt, "censored": False})

# print(data)
plot_recc_data(data)

In [ ]:
import glob, os
import pydicom

for i in glob.glob(r"F:\TCIA\Oropharyngeal-Radiomics-Outcomes\manifest-1585330520084\HNSCC\HNSCC*"):
    for j in glob.glob(os.path.join(i, "*")):
        for k in glob.glob(os.path.join(i, j, "*")):
            # print(os.listdir(k)[0])
            dcm = pydicom.dcmread(os.path.join(k, os.listdir(k)[0]))
            print(dcm.get((0x0008, 0x0060)).value)

In [ ]:
import glob, os
import pydicom

for i in glob.glob(r"F:\TCIA\RADCURE\manifest-1734616820458\RADCURE\RADCURE*"):
    for j in glob.glob(os.path.join(i, "*")):
        for k in glob.glob(os.path.join(i, j, "*")):
            # print(os.listdir(k)[0])
            dcm = pydicom.dcmread(os.path.join(k, os.listdir(k)[0]))
            print(dcm.get((0x0008, 0x0060)).value)

In [ ]:
import glob, os
import pydicom

for i in glob.glob(r"F:\TCIA\QIN-HEADNECK\manifest-1600133301170\QIN-HEADNECK\QIN-HEADNECK*"):
    for j in glob.glob(os.path.join(i, "*")):
        for k in glob.glob(os.path.join(i, j, "*")):
            # print(os.listdir(k)[0])
            dcm = pydicom.dcmread(os.path.join(k, os.listdir(k)[0]))
            print(dcm.get((0x0008, 0x0060)).value)

# center ID

In [ ]:
# Oropharyngeal-Radiomics-Outcomes (1)
#   M.D. Anderson Cancer Center (MDA)

# Head-Neck-CT-Atlas (1)
#   MD Anderson Cancer Center (MDA)

# QIN-HEADNECK (1)
#   University of Iowa Hospital and Clinics (UIHC)

# Head-Neck-PET-CT (4)
#   HGJ: Hôpital général juif
#   CHUS: Centre hospitalier universitaire de Sherbrooke
#   HMR: Hôpital Maisonneuve-Rosemont de Montréal
#   CHUM: Centre hospitalier de l’Université de Montréal

# RADCURE (1)
#   University Health Network (UHN)

# HNSCC-3DCT-RT (1)
#   University of Miami Miller School of Medicine (UMMSM)

# manufacturer & machine 

In [ ]:
import pydicom
import os, glob
import tqdm
import pandas

def get_dicom_folder(path_):
    if not os.path.isdir(path_):
        return []
    
    paths_return = []

    list_of_path = glob.glob(os.path.join(path_, "*"))
    if all(map(lambda i: i.endswith(".dcm"), list_of_path)):
        return [path_]
    else:
        for i in list_of_path:
            paths_return.extend(get_dicom_folder(i))
    
    # print(f" found {len(paths_return)} dicom folders")
    return paths_return

def plot_machine_dist(path_):
    df = []
    for dcm_path in tqdm.tqdm(get_dicom_folder(path_)):
        dcm_path = os.path.join(dcm_path, os.listdir(dcm_path)[0])
        dcm = pydicom.dcmread(dcm_path)

        try:
            modality = dcm.get((0x0008, 0x0060)).value
            if not modality == "CT":
                continue

            df.append({"manufacturer": dcm.get((0x0008, 0x0070)).value, "model name": dcm.get((0x0008, 0x1090)).value})
        except (PermissionError, AttributeError, TypeError):
            continue

    df = pandas.DataFrame(df)

    fig = go.Figure(data=[go.Pie(
        labels=df["manufacturer"].unique(), 
        values=[list(df["manufacturer"]).count(k) for k in df["manufacturer"].unique()],
        title="manufacturer")])
    fig.show()

    fig = go.Figure(data=[go.Pie(
        labels=df["model name"].unique(), 
        values=[list(df["model name"]).count(k) for k in df["model name"].unique()],
        title="model")])
    fig.show()

In [ ]:
plot_machine_dist(r"F:\TCIA\Head-Neck-CT-Atlas")

In [ ]:
plot_machine_dist(r"F:\TCIA\Head-Neck-PET-CT")

In [ ]:
plot_machine_dist(r"F:\TCIA\HNSCC-3DCT-RT")

In [ ]:
plot_machine_dist(r"F:\TCIA\Oropharyngeal-Radiomics-Outcomes")

In [ ]:
plot_machine_dist(r"F:\TCIA\QIN-HEADNECK")

In [ ]:
plot_machine_dist(r"F:\TCIA\RADCURE")

# generate clinical text

In [ ]:
# RADCURE
col = {
    "patient_id": "Patient ID number randomly assigned to each patient prior to anonymizing the DICOM PHI tag (0010,0020)",
    "Age": "Patient age, years",
    "Sex": "Patient sex, male or female",
    "ECOG PS": "ECOG Performance status scale - assessment of patient's functional status;GRADE	ECOG PERFORMANCE STATUS;0= Fully active, able to carry on all pre-disease performance without restriction;1=Restricted in physically strenuous activity but ambulatory and able to carry out work of a light or sedentary nature, e.g., light house work, office work;2=Ambulatory and capable of all selfcare but unable to carry out any work activities; up and about more than 50% of waking hours;3=Capable of only limited selfcare; confined to bed or chair more than 50% of waking hours;4=Completely disabled; cannot carry on any selfcare; totally confined to bed or chair;5=Dead",
    "Smoking PY": "Number of packs smoked in a year",
    "Smoking Status": "Smoking status at first consultation:Current smoker, ex-smoker, non-smoker",
    "Ds Site": "Primary cancer site",
    "Subsite": "Primary cancer subsite",
    "N": "AJCC 7th edition N category ",
    "Stage": "AJCC 7th edition stage groups",
    "Path": "Pathologic diagnosis/histology type",
    "HPV": "Tumor HPV status determined by p16 IHC +/- HPV DNA by PCR. Blank cell indicates no data available.",
    "Tx Modality": "How the surgery, radiation, and chemotherapy are combined - see mapping terminology",
    "Chemo": "Yes=received concurrent chemoradiotherapy, none=did not receive concurrent chemoradiotherapy",
    "Dose": "Total RT dose delivered during radiotherapy.",
    "Fx": "Number of RT fractions delivered."}

df = pandas.read_excel(r"F:\TCIA\RADCURE\RADCURE_Clinical_v04_20241219 (1).xlsx")
df = df[[c for c in df.columns if c in col.keys()]]
df = df.rename(columns=col, errors="raise")
df.to_csv(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\tmp\dfRADCURE.csv", index=False)

In [ ]:
# QIN-HEADNECK

col = {
    "PatientID": "PatientID",
    "Patient's Birth Date": "Patient's age",
    "Patient's Sex": "Patient's Sex",
    "Patient's Weight": "Patient's Weight",
    "Patient's Height": "Patient's Height",
    "History of Diabetes Mellitus": "History of Diabetes Mellitus",
    "Alcohol consumption": "Alcohol consumption",
    "Tobacco Smoking Behavior": "Tobacco Smoking Behavior",
    "N Stage": "N Stage",
    "Tumor staging": "Tumor staging",
    "Primary tumor site": "Primary tumor site",
    "Radiotherapy Procedure, Total radiation dose delivered": "Radiotherapy Procedure, Total radiation dose delivered",
    "Radiotherapy Procedure, Radiation dose per fraction": "Radiotherapy Procedure, Radiation dose per fraction",
}



df1 = pandas.read_excel(r"F:\TCIA\QIN-HEADNECK\Batch_01-and-Batch_02-Clinical-Data_aug242020.xlsx", header=0, sheet_name="uiowa_clinical_data_batch1_aug2")
df2 = pandas.read_excel(r"F:\TCIA\QIN-HEADNECK\Batch_01-and-Batch_02-Clinical-Data_aug242020.xlsx", header=0, sheet_name="uiowa_clinical_data_batch2_aug2")
df = pandas.concat((df1, df2)).drop(index=0)

# convert birth date to age based on diagnostic date
df["Patient's Birth Date"] = pandas.to_datetime(df["Diagnositic Procedure, Biopsy, Date of procedure"]).dt.year - df["Patient's Birth Date"]

df = df[[c for c in df.columns if c in col.keys()]]
df = df.rename(columns=col, errors="raise")
df.to_csv(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\tmp\dfQIN-HEADNECK.csv", index=False)

In [ ]:
# Oropharyngeal-Radiomics-Outcomes

col = {
    "TCIA Radiomics dummy ID of To_Submit_Final": "TCIA Radiomics dummy ID of To_Submit_Final",
    "Gender": "Gender",
    "Age at Diag": "Age at Diag",
    "Smoking status": "Smoking status",
    "Smoking status (Packs-Years)": "Smoking status (Packs-Years)",
    "Cancer subsite of origin": "Cancer subsite of origin",
    "HPV Status": "HPV Status",
    "N-category": "N-category",
    "AJCC Stage (7th edition)": "AJCC Stage (7th edition)",
    "Total prescribed Radiation treatment dose": "Total prescribed Radiation treatment dose",    
}

df = pandas.read_csv(r"F:\TCIA\Oropharyngeal-Radiomics-Outcomes\Radiomics_Outcome_Prediction_in_OPC_ASRM_corrected.csv")

df = df[[c for c in df.columns if c in col.keys()]]
df = df.rename(columns=col, errors="raise")
df.to_csv(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\tmp\dfOropharyngeal-Radiomics-Outcomes.csv", index=False)

In [ ]:
col = {
    "Patient #": "Patient #",
    "Sex": "Sex",
    "Age": "Age",
    "Primary Site": "Primary Site",
    "N-stage": "N-stage",
    "TNM group stage": "TNM group stage",
    "HPV status": "HPV status",
}

df = []
for i in ("HGJ", "CHUS", "HMR", "CHUM"):
    df__ = pandas.read_excel(r"F:\TCIA\Head-Neck-PET-CT\INFOclinical_HN_Version2_30may2018.xlsx", sheet_name=i)
    df.append(df__)
df = pandas.concat(df)


df = df[[c for c in df.columns if c in col.keys()]]
df = df.rename(columns=col, errors="raise")
df.to_csv(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\tmp\dfHeadNeckPETCT.csv", index=False)

In [ ]:
col = {
    "TCIA PatientID": "Patient ID number randomly assigned to each patient prior to anonymizing the DICOM PHI tag (0010,0020)",
    "Sex": "Patient sex, male or female",
    "Age": "Patient age, years",
    "Site": "Primary cancer subsite. CUP = cancer of unknown primary",
    "Histology": "Cancer histopathology. SCC=squamous cell carcinoma",
    "N": "AJCC 7th edition N stage",
    "Stage": "AJCC 7th edition summary stage",
    "RT Total Dose (Gy)": "Total RT dose delivered during radiotherapy.",
    "Dose/Fraction (Gy/fx)": "Dose delivered to prescription target volume (gross disease or post-operative tumor bed)",
    "Number of Fractions": "Number of RT fractions delivered.",
    "Smoking History": "Smoking history coded: 0=never smoker, 1= fewer than 10 pack-years, 2= 10 or more pack-years",
    "Current Smoker": "Current smoker: 0=no, 1=yes",
    "BMI start treat (kg/m2)": "Patient BMI at the start of radiotherapy.",
}

# Head-Neck-CT-Atlas
df = pandas.read_excel(r"F:\TCIA\Head-Neck-CT-Atlas\HNSCC-MDA-Data_update_20240514.xlsx", sheet_name="HNSCC-MDA-Data_update")

df = df[[c for c in df.columns if c in col.keys()]]
df = df.rename(columns=col, errors="raise")
df.to_csv(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\tmp\dfHead-Neck-CT-Atlas.csv", index=False)